> **SMOKE 版本（run_smoke.ipynb）**：训练/回测逻辑与 run.ipynb 完全一致，仅数据配置换成 2023-09-01~2024-02-29 smoke 窗口并读取 cache/diffsformer_smoke 缓存。用于在 Colab 上端到端验证管线；测试集仅 4 个交易日，结果无统计意义。

# Complete DiffsFormer reproduction

This notebook uses the settled `origin_extension.ipynb` benchmark unchanged as the target experiment: the same YMM point-in-time CSI300 data, purged split, Alpha158 normalization, downstream Transformer, daily IC, and Top30/Drop5 backtest.

The experimental treatment adds only: (1) DiffsFormer pre-training on a balanced all-A-share source sample, and (2) a newly edited **augmented-only** target-training sample in every downstream epoch. Target validation and test data are never augmented. Expensive data and model stages are fingerprinted, cached, and restartable.

**Run this notebook from a fresh CUDA kernel.** The executed benchmark holds several gigabytes of arrays in memory.

In [ ]:
import gc, hashlib, importlib, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy import stats
import matplotlib.pyplot as plt
from IPython.display import display

import alpha158 as alpha158_module
import diffsformer_model as model_module
import train_pipeline as pipeline_module
import diffsformer_data as data_module
for module in (alpha158_module, model_module, pipeline_module, data_module):
    importlib.reload(module)
from diffsformer_model import DiffsFormer
from diffsformer_data import prepare_target_data, prepare_source_data
from train_pipeline import (
    FactorBatch, TrainConfig, DiffsFormerTrainer,
    FactorAugmentor, DownstreamTrainer,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        f'CUDA is required. This kernel has {torch.__version__}; install/select a CUDA-enabled PyTorch kernel before continuing.'
    )
DEVICE = 'cuda'
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
CACHE_DIR = Path('cache/diffsformer_smoke')
CHECKPOINT_DIR = Path('checkpoints/diffsformer_smoke')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f'PyTorch {torch.__version__} | {torch.cuda.get_device_name(0)}')

In [ ]:
DATA_CONFIG = {
    # smoke 端到端验证版：与 cache/diffsformer_smoke 缓存指纹完全对齐（train 23100/val 681/test 3127）。
    'start_date': '2023-09-01', 'fetch_end_date': '2024-02-29',
    'train_end': '2024-01-31',
    'valid_start': '2024-02-01', 'valid_end': '2024-02-14',
    'test_start': '2024-02-19', 'test_end': '2024-02-22',
    'lookback': 8, 'label_horizon': 5, 'purge_sessions': 5,
}
SOURCE_CONFIG = {
    # smoke 源域：source_end = train_end = 行业快照日期（2024-01-31）。
    'source_start': '2023-09-01', 'source_end': DATA_CONFIG['train_end'],
    'lookback': 8, 'label_horizon': 5,
    'source_stride': 10, 'source_max_samples': 100_000,
    'source_chunk_size': 64, 'seed': SEED,
}
DM_CONFIG = {
    'hidden_dim': 256, 'num_layers': 6, 'num_heads': 8,
    'mlp_ratio': 4.0, 'dropout': 0.1, 'total_timesteps': 1000,
    'editing_step': 300, 'batch_size': 512, 'learning_rate': 1e-4,
    'weight_decay': 1e-5, 'epochs': 100, 'uncondition_prob': 0.1,
    'stop_loss': 0.965, 'min_epochs': 5, 'save_interval': 5,
}
AUGMENT_CONFIG = {
    'editing_step': 300, 'guidance_strength': 2.0,
    # The paper specifies DDIM acceleration but not l; 20 is the disclosed runtime choice.
    'ddim_steps': 20, 'micro_batch_size': 512,
    'loss_guided_noise': True, 'noise_boost_factor': 1.5,
}
DOWNSTREAM_CONFIG = {
    # Identical to origin_extension.
    'model': 'transformer', 'hidden_dim': 128,
    'num_layers': 2, 'num_heads': 4, 'batch_size': 1024,
    'learning_rate': 1e-4, 'epochs': 20, 'early_stopping': 4,
}
BACKTEST_CONFIG = {
    'top_n': 30, 'n_drop': 5, 'rebalance_every': 1, 'commission': 0.0003,
}
RUN_FINGERPRINT = hashlib.sha256(json.dumps(
    [DATA_CONFIG, SOURCE_CONFIG, DM_CONFIG, AUGMENT_CONFIG, DOWNSTREAM_CONFIG],
    sort_keys=True, default=str,
).encode()).hexdigest()[:16]
print('Run fingerprint:', RUN_FINGERPRINT)

In [ ]:
started = time.perf_counter()
target = prepare_target_data(DATA_CONFIG, CACHE_DIR)
assert target.train_features.shape[1:] == target.val_features.shape[1:] == target.test_features.shape[1:] == (8, 158)
assert target.train_industries.min() >= 0 and target.train_industries.max() < target.num_industries
print('Target fingerprint:', target.fingerprint)
print('Target metadata:', target.metadata)
print(f'Target loaded in {time.perf_counter() - started:.1f}s')

In [ ]:
started = time.perf_counter()
source = prepare_source_data(SOURCE_CONFIG, target, CACHE_DIR)
assert source.features.shape[1:] == (8, 158)
assert len(source.features) == len(source.labels) == len(source.industries)
assert source.industries.min() >= 0 and source.industries.max() < target.num_industries
print('Source fingerprint:', source.fingerprint)
print('Source metadata:', source.metadata)
print(f'Source loaded in {time.perf_counter() - started:.1f}s')

In [ ]:
def daily_model_ic(predictions, targets, dates):
    rows = []
    date_index = pd.DatetimeIndex(pd.to_datetime(dates))
    for date in pd.DatetimeIndex(pd.unique(date_index)).sort_values():
        mask = (date_index == date) & np.isfinite(predictions) & np.isfinite(targets)
        if mask.sum() < 20 or np.std(predictions[mask]) == 0 or np.std(targets[mask]) == 0:
            continue
        rows.append((
            date, stats.pearsonr(predictions[mask], targets[mask])[0],
            stats.spearmanr(predictions[mask], targets[mask])[0],
        ))
    if not rows:
        raise ValueError('No valid daily cross-sections for IC')
    return pd.DataFrame(rows, columns=['date', 'IC', 'RankIC']).set_index('date')

model_config = {
    'n_factors': 158, 'seq_len': 8,
    'hidden_dim': DOWNSTREAM_CONFIG['hidden_dim'],
    'num_layers': DOWNSTREAM_CONFIG['num_layers'],
    'num_heads': DOWNSTREAM_CONFIG['num_heads'],
}
baseline_path = CHECKPOINT_DIR / f'baseline_{RUN_FINGERPRINT}.pt'
baseline = DownstreamTrainer(DOWNSTREAM_CONFIG['model'])
if baseline_path.exists():
    baseline.load_direct_checkpoint(baseline_path, DEVICE, RUN_FINGERPRINT)
    print('Loaded baseline:', baseline_path)
else:
    np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    baseline.train_direct(
        target.train_features, target.train_labels,
        target.val_features, target.val_labels,
        hidden_dim=DOWNSTREAM_CONFIG['hidden_dim'],
        num_layers=DOWNSTREAM_CONFIG['num_layers'],
        num_heads=DOWNSTREAM_CONFIG['num_heads'],
        batch_size=DOWNSTREAM_CONFIG['batch_size'],
        learning_rate=DOWNSTREAM_CONFIG['learning_rate'],
        epochs=DOWNSTREAM_CONFIG['epochs'],
        early_stopping_patience=DOWNSTREAM_CONFIG['early_stopping'],
        device=DEVICE,
    )
    baseline.save_direct_checkpoint(baseline_path, model_config, RUN_FINGERPRINT)
baseline_predictions = baseline.predict(target.test_features, DEVICE, batch_size=8192)
baseline_ic = daily_model_ic(baseline_predictions, target.test_labels, target.test_dates)
print(f'Baseline IC={baseline_ic.IC.mean():.6f} | RankIC={baseline_ic.RankIC.mean():.6f}')

# The paper's loss-guided proxy: easy in-sample points receive stronger editing noise.
noise_scale_path = CACHE_DIR / f'loss_guided_scale_{RUN_FINGERPRINT}.npy'
if noise_scale_path.exists():
    noise_scale = np.load(noise_scale_path, mmap_mode='r+')
else:
    train_predictions = baseline.predict(target.train_features, DEVICE, batch_size=8192)
    train_loss = np.square(train_predictions - target.train_labels)
    low, high = np.quantile(train_loss, [0.05, 0.95])
    easy_score = 1.0 - np.clip((train_loss - low) / max(high - low, 1e-12), 0.0, 1.0)
    if AUGMENT_CONFIG['loss_guided_noise']:
        noise_scale = 1.0 + (AUGMENT_CONFIG['noise_boost_factor'] - 1.0) * easy_score
    else:
        noise_scale = np.ones_like(easy_score)
    np.save(noise_scale_path, noise_scale.astype(np.float32), allow_pickle=False)
    noise_scale = np.load(noise_scale_path, mmap_mode='r+')
print(f'Loss-guided noise scale: {noise_scale.min():.3f} to {noise_scale.max():.3f}')

In [ ]:
dm_final_path = CHECKPOINT_DIR / f'diffsformer_final_{RUN_FINGERPRINT}.pt'
if dm_final_path.exists():
    diffsformer = DiffsFormer.load(dm_final_path, DEVICE).to(DEVICE)
else:
    np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    diffsformer = DiffsFormer(
        seq_len=8, n_factors=158, hidden_dim=DM_CONFIG['hidden_dim'],
        num_layers=DM_CONFIG['num_layers'], num_heads=DM_CONFIG['num_heads'],
        mlp_ratio=DM_CONFIG['mlp_ratio'], dropout=DM_CONFIG['dropout'],
        total_timesteps=DM_CONFIG['total_timesteps'],
        num_industries=target.num_industries,
    )
    source_batch = FactorBatch(
        torch.from_numpy(source.features),
        torch.from_numpy(source.labels),
        torch.from_numpy(source.industries),
    )
    dm_trainer = DiffsFormerTrainer(
        diffsformer, source_batch,
        TrainConfig(
            editing_step=DM_CONFIG['editing_step'],
            batch_size=DM_CONFIG['batch_size'],
            learning_rate=DM_CONFIG['learning_rate'],
            weight_decay=DM_CONFIG['weight_decay'],
            epochs=DM_CONFIG['epochs'],
            uncondition_prob=DM_CONFIG['uncondition_prob'],
            use_loss_guided_noise=False,
            log_interval=1, save_interval=DM_CONFIG['save_interval'],
            stop_loss=DM_CONFIG['stop_loss'], min_epochs=DM_CONFIG['min_epochs'],
            amp=True, data_fingerprint=source.fingerprint,
        ),
        device=DEVICE, checkpoint_dir=CHECKPOINT_DIR / f'dm_{RUN_FINGERPRINT}',
    )
    internal_latest = CHECKPOINT_DIR / f'dm_{RUN_FINGERPRINT}' / 'diffsformer_latest.pt'
    diffsformer = dm_trainer.train(str(internal_latest) if internal_latest.exists() else None)
    diffsformer.save(dm_final_path)
diffsformer.eval()
del source
if 'source_batch' in globals(): del source_batch
gc.collect(); torch.cuda.empty_cache()

In [ ]:
augmentor = FactorAugmentor(
    diffsformer, editing_step=AUGMENT_CONFIG['editing_step'],
    guidance_strength=AUGMENT_CONFIG['guidance_strength'],
    ddim_steps=AUGMENT_CONFIG['ddim_steps'],
    batch_size=AUGMENT_CONFIG['micro_batch_size'], device=DEVICE, amp=True,
)
augmented_final_path = CHECKPOINT_DIR / f'augmented_final_{RUN_FINGERPRINT}.pt'
augmented_latest_path = CHECKPOINT_DIR / f'augmented_latest_{RUN_FINGERPRINT}.pt'
augmented = DownstreamTrainer(DOWNSTREAM_CONFIG['model'])
if augmented_final_path.exists():
    augmented.load_direct_checkpoint(augmented_final_path, DEVICE, RUN_FINGERPRINT)
    print('Loaded augmented model:', augmented_final_path)
else:
    # Paired initialization with the baseline seed. Validation/test remain original.
    np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    augmented.train_augmented(
        target.train_features, target.train_labels, target.train_industries,
        target.val_features, target.val_labels, augmentor,
        noise_scale=noise_scale,
        hidden_dim=DOWNSTREAM_CONFIG['hidden_dim'],
        num_layers=DOWNSTREAM_CONFIG['num_layers'],
        num_heads=DOWNSTREAM_CONFIG['num_heads'],
        batch_size=DOWNSTREAM_CONFIG['batch_size'],
        learning_rate=DOWNSTREAM_CONFIG['learning_rate'],
        epochs=DOWNSTREAM_CONFIG['epochs'],
        early_stopping_patience=DOWNSTREAM_CONFIG['early_stopping'],
        device=DEVICE, checkpoint_path=augmented_latest_path,
        resume=True, data_fingerprint=RUN_FINGERPRINT,
    )
    augmented.save_direct_checkpoint(augmented_final_path, model_config, RUN_FINGERPRINT)
augmented_predictions = augmented.predict(target.test_features, DEVICE, batch_size=8192)
augmented_ic = daily_model_ic(augmented_predictions, target.test_labels, target.test_dates)
print(f'Augmented IC={augmented_ic.IC.mean():.6f} | RankIC={augmented_ic.RankIC.mean():.6f}')

# Small fidelity diagnostic; this sample is not used for model selection.
diagnostic_rng = np.random.default_rng(SEED)
diagnostic_idx = diagnostic_rng.choice(len(target.train_features), 2048, replace=False)
edited_sample = augmentor.augment_tensor(
    torch.from_numpy(np.asarray(target.train_features[diagnostic_idx])),
    torch.from_numpy(np.asarray(target.train_labels[diagnostic_idx])),
    torch.from_numpy(np.asarray(target.train_industries[diagnostic_idx])),
    torch.from_numpy(np.asarray(noise_scale[diagnostic_idx])),
).cpu().numpy()
original_sample = np.asarray(target.train_features[diagnostic_idx])
display(pd.Series({
    'edit_RMSE': np.sqrt(np.mean(np.square(edited_sample - original_sample))),
    'edit_correlation': np.corrcoef(edited_sample.ravel(), original_sample.ravel())[0, 1],
    'edited_abs_max': np.abs(edited_sample).max(),
}).to_frame('value'))
rolling = pd.concat({
    'baseline': baseline_ic.IC, 'augmented': augmented_ic.IC,
}, axis=1).rolling(20, min_periods=5).mean()
ax = rolling.plot(figsize=(14, 4)); ax.axhline(0, color='black', lw=.7)
ax.set_title('20-day rolling daily IC: paired baseline vs DiffsFormer')
plt.tight_layout(); plt.show()

In [ ]:
def topk_dropout_backtest(
    predictions, realized_returns, dates, stocks, top_n=30, n_drop=5,
    rebalance_every=1, commission=0.0003,
):
    daily, turnover, raw_turnover, holdings = [], [], [], []
    previous, raw_previous = set(), set()
    date_index = pd.DatetimeIndex(pd.to_datetime(dates))
    for step, date in enumerate(pd.DatetimeIndex(pd.unique(date_index)).sort_values()):
        indices = np.flatnonzero((date_index == date) & np.isfinite(predictions))
        if len(indices) < top_n:
            continue
        by_stock = {stocks[i]: i for i in indices}
        ranked = sorted(by_stock, key=lambda s: predictions[by_stock[s]], reverse=True)
        available = set(ranked)
        raw_selected = set(ranked[:top_n])
        raw_one_way = 1.0 if not raw_previous else 1 - len(raw_selected & raw_previous) / top_n
        raw_previous = raw_selected
        if not previous:
            selected = set(ranked[:top_n])
        else:
            selected = previous & available
            for stock in ranked:
                if len(selected) >= top_n: break
                selected.add(stock)
            if step % rebalance_every == 0:
                for _ in range(n_drop):
                    best_new = next((s for s in ranked if s not in selected), None)
                    if best_new is None: break
                    worst_old = min(selected, key=lambda s: predictions[by_stock[s]])
                    if predictions[by_stock[best_new]] <= predictions[by_stock[worst_old]]: break
                    selected.remove(worst_old); selected.add(best_new)
        selected_indices = np.array([by_stock[s] for s in selected], dtype=int)
        gross = float(np.nan_to_num(realized_returns[selected_indices], nan=0.0).mean())
        if not previous:
            one_way, cost = 1.0, commission
        else:
            union = selected | previous
            old_weight, new_weight = 1 / len(previous), 1 / len(selected)
            one_way = 0.5 * sum(abs(
                (new_weight if s in selected else 0.0) -
                (old_weight if s in previous else 0.0)
            ) for s in union)
            cost = 2 * commission * one_way
        daily.append((date, gross - cost)); turnover.append((date, one_way))
        raw_turnover.append((date, raw_one_way))
        holdings.append({'date': date, 'stocks': sorted(selected)})
        previous = selected
    returns = pd.Series(dict(daily), name='strategy_return').sort_index()
    turns = pd.Series(dict(turnover), name='turnover').sort_index()
    raw_turns = pd.Series(dict(raw_turnover), name='raw_top30_turnover').sort_index()
    wealth = (1 + returns).cumprod()
    running_max = pd.Series(np.maximum.accumulate(np.r_[1.0, wealth.values])[1:], index=wealth.index)
    drawdown = wealth / running_max - 1
    vol = returns.std(ddof=1)
    metrics = {
        'sessions': len(returns), 'total_return': wealth.iloc[-1] - 1,
        'annual_return': wealth.iloc[-1] ** (252 / len(returns)) - 1,
        'annual_vol': vol * np.sqrt(252),
        'sharpe': returns.mean() / vol * np.sqrt(252) if vol > 0 else np.nan,
        'max_drawdown': drawdown.min(), 'mean_turnover': turns.iloc[1:].mean(),
        'raw_top30_mean_turnover': raw_turns.iloc[1:].mean(),
    }
    return {'returns': returns, 'wealth': wealth, 'turnover': turns, 'raw_turnover': raw_turns, 'holdings': holdings, **metrics}

experiments = {
    'baseline': (baseline_predictions, baseline_ic),
    'DiffsFormer': (augmented_predictions, augmented_ic),
}
backtests = {name: topk_dropout_backtest(
    predictions, target.test_returns, target.test_dates, target.test_instruments,
    **BACKTEST_CONFIG,
) for name, (predictions, _) in experiments.items()}
summary = pd.DataFrame([{
    'experiment': name, 'IC': ic.IC.mean(), 'RankIC': ic.RankIC.mean(),
    **{key: backtests[name][key] for key in (
        'sessions', 'total_return', 'annual_return', 'annual_vol',
        'sharpe', 'max_drawdown', 'mean_turnover', 'raw_top30_mean_turnover',
    )},
} for name, (_, ic) in experiments.items()]).set_index('experiment')
display(summary)
display((summary.loc['DiffsFormer'] - summary.loc['baseline']).to_frame('DiffsFormer minus baseline'))

benchmark = pd.Series(
    np.asarray(target.benchmark_returns),
    index=pd.to_datetime(np.asarray(target.benchmark_dates)),
).reindex(backtests['baseline']['returns'].index).fillna(0.0)
benchmark_wealth = (1 + benchmark).cumprod()
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for name, result in backtests.items():
    axes[0].plot(result['wealth'].index, result['wealth'], label=name)
axes[0].plot(benchmark_wealth.index, benchmark_wealth, label='CSI300', alpha=.8)
axes[0].axhline(1, color='black', lw=.7); axes[0].legend(); axes[0].set_title('Net equity curves')
for name, result in backtests.items():
    peak = pd.Series(np.maximum.accumulate(np.r_[1.0, result['wealth'].values])[1:], index=result['wealth'].index)
    axes[1].plot(result['wealth'].index, result['wealth'] / peak - 1, label=name)
axes[1].legend(); axes[1].set_title('Drawdown')
for name, result in backtests.items():
    axes[2].plot(result['turnover'].index, result['turnover'].rolling(20, min_periods=1).mean(), label=name)
axes[2].legend(); axes[2].set_title('20-day mean one-way turnover')
for ax in axes: ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig('diffsformer_smoke_results.png', dpi=150); plt.show()
print(f"Test curve: {summary.sessions.iloc[0]:.0f} sessions, {backtests['baseline']['returns'].index[0].date()} to {backtests['baseline']['returns'].index[-1].date()}")